In [1]:
# ============================================================================
# COMPLETE TABTRANSFORMER PIPELINE - FROM DATA LOADING TO MODEL TRAINING
# Each section is a separate cell - copy into your Jupyter notebook
# ============================================================================


"""
================================================================================
CELL 1: Install Required Libraries
================================================================================
"""
!pip install boto3
!pip install pandas
!pip install tensorflow
!pip install scikit-learn

print("✓ All packages installed!")


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
✓ All packages installed!


In [2]:
"""
================================================================================
CELL 2: Load Dataset from RunPod S3
================================================================================
"""
import boto3
import pandas as pd
from botocore.config import Config
from io import BytesIO

print("Loading dataset from RunPod S3...")

# ---- RunPod S3 location ----
BUCKET = "e9tcw5eupu"
KEY = "data/eff_training.csv"
ENDPOINT = "https://s3api-eu-ro-1.runpod.io"
REGION = "eu-ro-1"

# ---- Your RunPod S3 credentials ----
ACCESS_KEY = "user_37sKcYrvnk9UXaIY3B3Zr90MH0g"
SECRET_KEY = "rps_YW72UMRXEMRVC8A407OCL08J8G34U1B3QTNO1ETX18pa1n"

cfg = Config(
    region_name=REGION,
    signature_version="s3v4",
    s3={"addressing_style": "path"},
)

s3 = boto3.client(
    "s3",
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
    endpoint_url=ENDPOINT,
    config=cfg,
)

# Load CSV from RunPod S3 into a variable named `data`
obj = s3.get_object(Bucket=BUCKET, Key=KEY)
data = pd.read_csv(BytesIO(obj["Body"].read()))

print(f"✓ Data loaded successfully!")
print(f"  Data type: {type(data)}")
print(f"  Data shape: {data.shape}")
print(f"\nFirst few rows:")
print(data.head())


Loading dataset from RunPod S3...
✓ Data loaded successfully!
  Data type: <class 'pandas.DataFrame'>
  Data shape: (6134, 5138)

First few rows:
                           photo_id        f1        f2        f3        f4  \
0  6ab1d061f51c6079633aeceed2faeb0b  0.000068  0.108145 -0.138813  0.633156   
1  e94e2e05fb8b099955bbc4fa5ce81e22  0.020843  0.026005 -0.093442  0.736929   
2  ba6951a4f37fc9302243370e927a02e2  0.014542 -0.071332 -0.154407  0.577781   
3  947d16539d4702427aa74f737329ffb9  0.041775  0.075746 -0.128497  0.485010   
4  9326695bf62926ec22690f576a633bba  0.004397  0.058590 -0.154224  0.528140   

         f5        f6        f7        f8        f9  ...         hip  \
0  0.346266 -0.046055  0.016021 -0.058632  0.097968  ...  105.333900   
1  0.240569  0.089982 -0.112391  0.000435 -0.076110  ...  101.478989   
2  0.196485 -0.125341 -0.056713 -0.027295  0.094879  ...   97.488243   
3  0.120409  0.011227  0.017852 -0.089796 -0.011273  ...  120.586845   
4  0.290956 -0.1084

In [3]:
"""
================================================================================
CELL 3: Categorical Encoding for Gender
================================================================================
"""
import pandas as pd

print("Encoding gender feature...")

# Convert 'gender' to categorical and then to numeric codes
data['gender'] = data['gender'].astype('category')
data['gender'] = data['gender'].cat.codes

print(f"✓ Gender encoded!")
print(f"  Unique values: {data['gender'].unique()}")
print(f"  Value counts:\n{data['gender'].value_counts()}")

Encoding gender feature...
✓ Gender encoded!
  Unique values: [0 1]
  Value counts:
gender
1    3650
0    2484
Name: count, dtype: int64


In [4]:
"""
================================================================================
CELL 4: Define Weight Frequencies for Weight_kg Classes
================================================================================
"""
import numpy as np

print("Calculating class weights for weight_kg...")
print(f"Total samples in dataset: {len(data)}")

# Create boolean masks for the three weight_kg classes
class_1_mask = data['weight_kg'] < 60
class_2_mask = data['weight_kg'] > 100
class_3_mask = (data['weight_kg'] >= 60) & (data['weight_kg'] <= 100)

# Calculate class frequencies
freq_class_1 = class_1_mask.sum()
freq_class_2 = class_2_mask.sum()
freq_class_3 = class_3_mask.sum()

print(f"\n✓ Class frequencies:")
print(f"  Class 1 (weight_kg < 60): {freq_class_1}")
print(f"  Class 2 (weight_kg > 100): {freq_class_2}")
print(f"  Class 3 (60 <= weight_kg <= 100): {freq_class_3}")

# Number of classes
num_classes = 3
print(f"\n✓ Number of classes: {num_classes}")

# Compute inverse-frequency weights
total_samples = len(data)

def safe_weight(class_freq):
    if class_freq == 0:
        return np.nan
    return total_samples / (num_classes * class_freq)

weight_class_1 = safe_weight(freq_class_1)
weight_class_2 = safe_weight(freq_class_2)
weight_class_3 = safe_weight(freq_class_3)

print(f"\n✓ Class weights (inverse frequency):")
print(f"  Weight for Class 1 (weight_kg < 60): {weight_class_1:.4f}")
print(f"  Weight for Class 2 (weight_kg > 100): {weight_class_2:.4f}")
print(f"  Weight for Class 3 (60 <= weight_kg <= 100): {weight_class_3:.4f}")


Calculating class weights for weight_kg...
Total samples in dataset: 6134

✓ Class frequencies:
  Class 1 (weight_kg < 60): 1049
  Class 2 (weight_kg > 100): 514
  Class 3 (60 <= weight_kg <= 100): 4571

✓ Number of classes: 3

✓ Class weights (inverse frequency):
  Weight for Class 1 (weight_kg < 60): 1.9492
  Weight for Class 2 (weight_kg > 100): 3.9780
  Weight for Class 3 (60 <= weight_kg <= 100): 0.4473


In [5]:
"""
================================================================================
CELL 5: Define Weight Frequencies for Gender Classes
================================================================================
"""
print("Calculating class weights for gender...")

# Calculate class frequencies for gender
gender_counts = data['gender'].value_counts()

print(f"\n✓ Class frequencies for gender:")
for gender_class, freq in gender_counts.items():
    print(f"  Class {gender_class}: {freq}")

# Number of gender classes
num_gender_classes = len(gender_counts)
print(f"\n✓ Number of gender classes: {num_gender_classes}")

# Compute inverse-frequency weights for each gender class
gender_weights = {}
for gender_class, freq in gender_counts.items():
    gender_weights[gender_class] = safe_weight(freq)

print(f"\n✓ Class weights (inverse frequency) for gender:")
for gender_class, weight in gender_weights.items():
    print(f"  Weight for class {gender_class}: {weight:.4f}")


Calculating class weights for gender...

✓ Class frequencies for gender:
  Class 1: 3650
  Class 0: 2484

✓ Number of gender classes: 2

✓ Class weights (inverse frequency) for gender:
  Weight for class 1: 0.5602
  Weight for class 0: 0.8231


In [6]:
"""
================================================================================
CELL 6: Combine Weight Class and Gender Class Weights
================================================================================
"""
print("Combining weight-class and gender-class weights...")

# Store weight-class weights
weight_class_weights = {
    'weight_<60': weight_class_1,
    'weight_>100': weight_class_2,
    'weight_60_100': weight_class_3
}

print(f"\n✓ Weight-class weights:")
for key, val in weight_class_weights.items():
    print(f"  {key}: {val:.4f}")

print(f"\n✓ Gender-class weights:")
for key, val in gender_weights.items():
    print(f"  Class {key}: {val:.4f}")

# Multiply each gender class with each weight class
combined_weights = {}

print(f"\n✓ Combined weights for each (weight_class, gender_class):")
for w_label, w_w in weight_class_weights.items():
    for g_label, w_g in gender_weights.items():
        wi = w_w * w_g
        combined_weights[(w_label, g_label)] = wi
        print(f"  {w_label} & gender {g_label}: {wi:.4f}")


Combining weight-class and gender-class weights...

✓ Weight-class weights:
  weight_<60: 1.9492
  weight_>100: 3.9780
  weight_60_100: 0.4473

✓ Gender-class weights:
  Class 1: 0.5602
  Class 0: 0.8231

✓ Combined weights for each (weight_class, gender_class):
  weight_<60 & gender 1: 1.0919
  weight_<60 & gender 0: 1.6044
  weight_>100 & gender 1: 2.2284
  weight_>100 & gender 0: 3.2744
  weight_60_100 & gender 1: 0.2506
  weight_60_100 & gender 0: 0.3682


In [7]:
"""
================================================================================
CELL 7: Create Index Column and Weight Dictionary
================================================================================
"""
import pickle

print("Creating index column and weight dictionary...")

# Add index column
data['index'] = range(len(data))

# Move 'index' to the front
cols = ['index'] + [c for c in data.columns if c != 'index']
data = data[cols]

print(f"✓ Index column added!")
print(f"  Columns: {list(data.columns[:5])}...")

# Helper function to get weight class label
def get_weight_class(w):
    if w < 60:
        return 'weight_<60'
    elif w > 100:
        return 'weight_>100'
    else:
        return 'weight_60_100'

# Build final_weights dictionary
final_weights = {}

print(f"\n✓ Building final_weights dictionary...")

for _, row in data.iterrows():
    idx_val = row['index']
    gender_val = row['gender']
    weight_val = row['weight_kg']
    
    w_class = get_weight_class(weight_val)
    w_weight = weight_class_weights[w_class]
    w_gender = gender_weights[gender_val]
    
    combined_w = w_weight * w_gender
    final_weights[idx_val] = combined_w

print(f"✓ Dictionary created with {len(final_weights)} entries")
print(f"  First 5 items: {list(final_weights.items())[:5]}")

# Check index 0
print(f"\n✓ Checking entry with index 0:")
row0 = data.loc[data['index'] == 0].iloc[0]
gender0 = row0['gender']
weight0 = row0['weight_kg']
w_class0 = get_weight_class(weight0)

print(f"  Gender: {gender0}")
print(f"  Weight_kg: {weight0}")
print(f"  Weight class: {w_class0}")
print(f"  Combined weight: {final_weights[0]:.4f}")

# Save final_weights dictionary
print(f"\n✓ Saving final_weights.pkl...")
with open('final_weights.pkl', 'wb') as f:
    pickle.dump(final_weights, f)
print(f"✓ Dictionary saved!")


Creating index column and weight dictionary...
✓ Index column added!
  Columns: ['index', 'photo_id', 'f1', 'f2', 'f3']...

✓ Building final_weights dictionary...
✓ Dictionary created with 6134 entries
  First 5 items: [(0, np.float64(0.3681986747807079)), (1, np.float64(0.25057685154939136)), (2, np.float64(0.25057685154939136)), (3, np.float64(0.3681986747807079)), (4, np.float64(0.25057685154939136))]

✓ Checking entry with index 0:


/tmp/ipykernel_6360/2764810991.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data['index'] = range(len(data))


  Gender: 0
  Weight_kg: 72.0
  Weight class: weight_60_100
  Combined weight: 0.3682

✓ Saving final_weights.pkl...
✓ Dictionary saved!


In [8]:
"""
================================================================================
CELL 8: Apply Scaling to Features
================================================================================
"""
import os
import pickle
from sklearn.preprocessing import StandardScaler, RobustScaler

print("Applying scaling to features...")

# Columns to exclude from scaling
exclude_cols = ['photo_id', 'subject_id', 'index', 'gender']

# Standard-scaled feature set
standard_cols = [
    'ankle', 'arm-length', 'bicep', 'calf', 'chest', 'forearm', 'hip',
    'leg-length', 'shoulder-breadth', 'shoulder-to-crotch', 'thigh',
    'waist', 'wrist', 'weight_kg', 'height_cm'
]
standard_cols = [c for c in standard_cols if c in data.columns]

# Separate height from other standard features
height_col = "height_cm"
height_cols = [height_col] if height_col in data.columns else []
target_cols = [c for c in standard_cols if c != height_col]

# Robust-scale everything else
robust_cols = [
    c for c in data.columns
    if c not in exclude_cols and c not in target_cols and c not in height_cols
]

print(f"\n✓ Feature groups:")
print(f"  Height features: {len(height_cols)}")
print(f"  Standard features: {len(target_cols)}")
print(f"  Robust features: {len(robust_cols)}")

# Create scalers
height_scaler = StandardScaler()
standard_scaler = StandardScaler()
robust_scaler = RobustScaler()

# Fit and transform
if height_cols:
    data[height_cols] = height_scaler.fit_transform(data[height_cols])
    print(f"✓ Height scaled")

if target_cols:
    data[target_cols] = standard_scaler.fit_transform(data[target_cols])
    print(f"✓ Standard features scaled")

if robust_cols:
    data[robust_cols] = robust_scaler.fit_transform(data[robust_cols])
    print(f"✓ Robust features scaled")

# Save scalers
def save_pickle(obj, filename: str):
    path = os.path.join(os.getcwd(), filename)
    with open(path, "wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
    return path

if height_cols:
    save_pickle(height_scaler, "scaler_height_cm.pkl")
    print(f"✓ Saved: scaler_height_cm.pkl")

if target_cols:
    save_pickle(standard_scaler, "scaler_standard_features.pkl")
    print(f"✓ Saved: scaler_standard_features.pkl")

if robust_cols:
    save_pickle(robust_scaler, "scaler_robust_features.pkl")
    print(f"✓ Saved: scaler_robust_features.pkl")


Applying scaling to features...

✓ Feature groups:
  Height features: 1
  Standard features: 14
  Robust features: 5120
✓ Height scaled
✓ Standard features scaled
✓ Robust features scaled
✓ Saved: scaler_height_cm.pkl
✓ Saved: scaler_standard_features.pkl
✓ Saved: scaler_robust_features.pkl


In [9]:
"""
CELL 9 (CORRECTED): Split Data into X and Y
"""
print("Splitting data into features (X) and targets (Y)...")

# Target columns (dependent variables) 
target_cols = [
    'ankle', 'arm-length', 'bicep', 'calf', 'chest', 'forearm', 'hip',
    'leg-length', 'shoulder-breadth', 'shoulder-to-crotch', 'thigh',
    'waist', 'wrist', 'weight_kg'
]

Y = data[target_cols]
print(f"✓ Target columns: {len(target_cols)}")
print(f"  Y shape: {Y.shape}")

# Drop only IDs and index - KEEP gender and all other features
drop_cols = ['photo_id', 'subject_id', 'index'] + target_cols
X = data.drop(columns=drop_cols)

print(f"✓ Features (X) created")
print(f"  X shape: {X.shape}")
print(f"  'gender' in X: {'gender' in X.columns}")  # Should be True!

Splitting data into features (X) and targets (Y)...
✓ Target columns: 14
  Y shape: (6134, 14)
✓ Features (X) created
  X shape: (6134, 5122)
  'gender' in X: True


In [10]:
"""
================================================================================
CELL 10: Convert Sample Weights Dictionary to Array
================================================================================
"""
print("Converting sample weights to array...")

# Load final_weights.pkl
with open('final_weights.pkl', 'rb') as f:
    final_weights_dict = pickle.load(f)

print(f"✓ Loaded final_weights_dict with {len(final_weights_dict)} entries")

# Build sample_weight array based on DataFrame 'index' column
sample_weights = data['index'].map(final_weights_dict).values.astype('float32')

print(f"✓ Sample weights array created")
print(f"  Shape: {sample_weights.shape}")
print(f"  First 10 weights: {sample_weights[:10]}")


Converting sample weights to array...
✓ Loaded final_weights_dict with 6134 entries
✓ Sample weights array created
  Shape: (6134,)
  First 10 weights: [0.36819866 0.25057685 0.25057685 0.36819866 0.25057685 0.36819866
 0.25057685 0.25057685 0.36819866 0.25057685]


In [11]:
"""
================================================================================
CELL 11: Train/Validation Split
================================================================================
"""
from sklearn.model_selection import train_test_split

print("Splitting into train and validation sets...")

X_train, X_val, Y_train, Y_val, w_train, w_val = train_test_split(
    X, Y, sample_weights,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

print(f"✓ Data split complete!")
print(f"\nTraining set:")
print(f"  X_train shape: {X_train.shape}")
print(f"  Y_train shape: {Y_train.shape}")
print(f"  w_train shape: {w_train.shape}")

print(f"\nValidation set:")
print(f"  X_val shape: {X_val.shape}")
print(f"  Y_val shape: {Y_val.shape}")
print(f"  w_val shape: {w_val.shape}")

Splitting into train and validation sets...
✓ Data split complete!

Training set:
  X_train shape: (4907, 5122)
  Y_train shape: (4907, 14)
  w_train shape: (4907,)

Validation set:
  X_val shape: (1227, 5122)
  Y_val shape: (1227, 14)
  w_val shape: (1227,)


In [12]:
"""
================================================================================
CELL 12: Disable GPU (Optional - Use CPU Only)
================================================================================
"""
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

print("✓ GPU disabled - using CPU only")
print("  (Remove this cell if you want to use GPU)")


✓ GPU disabled - using CPU only
  (Remove this cell if you want to use GPU)


In [13]:
pip install matplotlib


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [14]:
# =========================
# CELL 13: Imports
# =========================

import numpy as np                     # numerical operations
import tensorflow as tf                # deep learning framework
from tensorflow import keras           # Keras API
from tensorflow.keras import layers    # neural network layers
from tensorflow.keras.models import Model  # model class
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau  # training callbacks
from tensorflow.keras.optimizers import Adam  # optimizer
import matplotlib.pyplot as plt        # plotting

print("✓ TensorFlow version:", tf.__version__)      # track TF version
print("✓ Libraries imported for FT-Transformer")    # progress log

2026-02-16 08:52:29.362847: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


✓ TensorFlow version: 2.20.0
✓ Libraries imported for FT-Transformer


In [15]:
# =========================
# CELL 14: Feature Tokenizer (FT-Transformer) with numeric token compression
# =========================
# - Numerical tokens: first compress many numeric features -> K values, then tokenize
# - Categorical tokens: Embedding(cat) + bias
# - Adds learnable [CLS] token

@tf.keras.utils.register_keras_serializable()
class FeatureTokenizer(layers.Layer):
    def __init__(self, num_numeric_features, categorical_cardinalities, embed_dim, numeric_token_count=32, **kwargs):
        super().__init__(**kwargs)                                      # init base layer
        self.num_numeric_features = num_numeric_features                # number of numeric features
        self.categorical_cardinalities = categorical_cardinalities      # list of vocab sizes
        self.embed_dim = embed_dim                                      # token dim
        self.numeric_token_count = numeric_token_count                  # K numeric tokens

        self.num_categorical_features = len(categorical_cardinalities)  # number of categorical features

        # One embedding table per categorical feature
        self.cat_embeddings = []                                        # store embedding layers
        for i, card in enumerate(categorical_cardinalities):            # loop categorical features
            self.cat_embeddings.append(
                layers.Embedding(
                    input_dim=card,                                     # vocab size
                    output_dim=embed_dim,                                # embedding dim
                    name=f"cat_embedding_{i}"                            # name
                )
            )

        # Compress numeric features from (n_num) -> (K)
        self.numeric_compressor = layers.Dense(
            units=numeric_token_count,                                   # output size K
            use_bias=True,                                               # include bias
            name="numeric_compressor"                                    # name
        )

    def build(self, input_shape):
        # Learnable tokenization weights for K numeric tokens: (K, D)
        self.W_num = self.add_weight(
            name="W_num_tok",                                           # name
            shape=(self.numeric_token_count, self.embed_dim),           # (K, D)
            initializer="random_normal",                                # init
            trainable=True                                              # trainable
        )

        # Learnable bias for K numeric tokens: (K, D)
        self.b_num = self.add_weight(
            name="b_num_tok",                                           # name
            shape=(self.numeric_token_count, self.embed_dim),           # (K, D)
            initializer="zeros",                                        # init
            trainable=True                                              # trainable
        )

        # Bias for categorical tokens: (n_cat, D)
        self.b_cat = self.add_weight(
            name="b_cat",                                               # name
            shape=(self.num_categorical_features, self.embed_dim),      # (n_cat, D)
            initializer="zeros",                                        # init
            trainable=True                                              # trainable
        )

        # Learnable [CLS] token: (1,1,D)
        self.cls_token = self.add_weight(
            name="cls_token",                                           # name
            shape=(1, 1, self.embed_dim),                               # (1,1,D)
            initializer="random_normal",                                # init
            trainable=True                                              # trainable
        )

        super().build(input_shape)                                      # finalize build

    def call(self, inputs, training=False):
        categorical_inputs, continuous_inputs = inputs                  # unpack inputs
        batch_size = tf.shape(continuous_inputs)[0]                     # get batch size

        # -------------------------
        # NUMERIC: compress -> tokenize
        # -------------------------
        z = self.numeric_compressor(continuous_inputs)                  # (B, K)
        z = tf.expand_dims(z, axis=-1)                                  # (B, K, 1)

        W = tf.expand_dims(self.W_num, axis=0)                          # (1, K, D)
        b = tf.expand_dims(self.b_num, axis=0)                          # (1, K, D)

        num_tokens = z * W + b                                          # (B, K, D)

        # -------------------------
        # CATEGORICAL: embedding + bias
        # -------------------------
        cat_tokens_list = []                                            # store cat tokens
        for i in range(self.num_categorical_features):                  # loop cat features
            cat_i = categorical_inputs[:, i:i+1]                        # (B,1)
            emb_i = self.cat_embeddings[i](cat_i)                       # (B,1,D)

            bias_i = tf.reshape(self.b_cat[i], (1, 1, self.embed_dim))   # (1,1,D)
            emb_i = emb_i + bias_i                                      # add bias

            cat_tokens_list.append(emb_i)                               # append token

        if self.num_categorical_features > 0:                           # if categorical exist
            cat_tokens = tf.concat(cat_tokens_list, axis=1)             # (B, n_cat, D)
        else:
            cat_tokens = None                                           # none

        # -------------------------
        # CLS token
        # -------------------------
        cls = tf.tile(self.cls_token, [batch_size, 1, 1])               # (B,1,D)

        # -------------------------
        # Final sequence: [CLS] + numeric(K) + categorical
        # -------------------------
        if cat_tokens is not None:
            tokens = tf.concat([cls, num_tokens, cat_tokens], axis=1)   # (B, 1+K+n_cat, D)
        else:
            tokens = tf.concat([cls, num_tokens], axis=1)               # (B, 1+K, D)

        return tokens                                                   # return tokens

    def get_config(self):
        config = super().get_config()                                   # base config
        config.update({
            "num_numeric_features": self.num_numeric_features,          # save
            "categorical_cardinalities": self.categorical_cardinalities,# save
            "embed_dim": self.embed_dim,                                # save
            "numeric_token_count": self.numeric_token_count             # save
        })
        return config


print("✓ FeatureTokenizer defined (with numeric token compression)")     # log

✓ FeatureTokenizer defined (with numeric token compression)


In [16]:
# =========================
# CELL 15: FT-Transformer Block (PreNorm + Residual) with proper build()
# =========================

@tf.keras.utils.register_keras_serializable()
class FTTransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout_rate=0.1, attention_dropout=0.1, use_attention_norm=True, **kwargs):
        super().__init__(**kwargs)                                      # init base
        self.embed_dim = embed_dim                                      # token dim
        self.num_heads = num_heads                                      # heads
        self.ff_dim = ff_dim                                            # FF dim
        self.dropout_rate = dropout_rate                                # dropout
        self.attention_dropout = attention_dropout                      # attn dropout
        self.use_attention_norm = use_attention_norm                    # FT tweak

        projection_dim = embed_dim // num_heads                         # per-head dim

        self.ln1 = layers.LayerNormalization(epsilon=1e-6, name="ln1")   # LN for attn
        self.mha = layers.MultiHeadAttention(
            num_heads=num_heads,                                        # heads
            key_dim=projection_dim,                                     # key dim
            dropout=attention_dropout,                                  # dropout
            name="mha"                                                  # name
        )
        self.drop1 = layers.Dropout(dropout_rate, name="drop1")         # dropout

        self.ln2 = layers.LayerNormalization(epsilon=1e-6, name="ln2")   # LN for FFN
        self.dense1 = layers.Dense(ff_dim, activation="relu", name="ffn_dense1")  # FFN
        self.drop2 = layers.Dropout(dropout_rate, name="drop2")         # dropout
        self.dense2 = layers.Dense(embed_dim, name="ffn_dense2")        # back to D
        self.drop3 = layers.Dropout(dropout_rate, name="drop3")         # dropout

    def build(self, input_shape):
        # input_shape: (B, L, D)
        self.ln1.build(input_shape)                                     # build ln1
        self.ln2.build(input_shape)                                     # build ln2
        self.dense1.build(input_shape)                                  # build dense1
        self.dense2.build((input_shape[0], input_shape[1], self.ff_dim))# build dense2
        super().build(input_shape)                                      # finalize

    def call(self, x, training=False):
        # ---- Attention (PreNorm) ----
        if self.use_attention_norm:
            x_norm = self.ln1(x)                                        # LN before attn
        else:
            x_norm = x                                                  # skip LN in first block

        attn_out = self.mha(query=x_norm, key=x_norm, value=x_norm, training=training)  # attn
        attn_out = self.drop1(attn_out, training=training)              # dropout
        x = x + attn_out                                                # residual

        # ---- FFN (PreNorm) ----
        x_norm2 = self.ln2(x)                                           # LN before FFN
        ffn_out = self.dense1(x_norm2)                                  # FFN
        ffn_out = self.drop2(ffn_out, training=training)                # dropout
        ffn_out = self.dense2(ffn_out)                                  # project
        ffn_out = self.drop3(ffn_out, training=training)                # dropout
        x = x + ffn_out                                                 # residual

        return x                                                        # output

    def get_config(self):
        config = super().get_config()                                   # base config
        config.update({
            "embed_dim": self.embed_dim,
            "num_heads": self.num_heads,
            "ff_dim": self.ff_dim,
            "dropout_rate": self.dropout_rate,
            "attention_dropout": self.attention_dropout,
            "use_attention_norm": self.use_attention_norm
        })
        return config


print("✓ FTTransformerBlock defined (with build())")                     # log

✓ FTTransformerBlock defined (with build())


In [17]:
# =========================
# CELL 16: Build FT-Transformer Model (uses CLS for prediction)
# =========================

def build_fttransformer(
    num_categorical_features,                                           # cat count
    num_continuous_features,                                            # numeric feature count (still used for input)
    categorical_cardinalities,                                          # vocab sizes
    embedding_dim=64,                                                   # token dim
    num_transformer_blocks=3,                                           # depth
    num_heads=8,                                                        # heads
    ff_dim=256,                                                         # FF dim
    mlp_hidden_units=[256, 128],                                        # head MLP
    dropout_rate=0.1,                                                   # dropout
    attention_dropout=0.1,                                              # attention dropout
    numeric_token_count=32,                                             # K numeric tokens (KEY FIX)
    num_outputs=14                                                      # outputs
):
    print("\n🏗 Building FT-Transformer...")                              # log

    categorical_input = layers.Input(                                   # cat input
        shape=(num_categorical_features,),                              # shape
        dtype=tf.int32,                                                 # dtype
        name="categorical_input"                                        # name
    )

    continuous_input = layers.Input(                                    # numeric input
        shape=(num_continuous_features,),                               # shape
        dtype=tf.float32,                                               # dtype
        name="continuous_input"                                         # name
    )

    print("  ✓ Tokenizing features...")                                  # log
    tokens = FeatureTokenizer(
        num_numeric_features=num_continuous_features,                   # numeric feature count
        categorical_cardinalities=categorical_cardinalities,            # vocab sizes
        embed_dim=embedding_dim,                                        # token dim
        numeric_token_count=numeric_token_count,                        # K tokens
        name="feature_tokenizer"                                        # name
    )([categorical_input, continuous_input])                            # token sequence

    x = tokens                                                          # init
    for i in range(num_transformer_blocks):                             # blocks
        use_attention_norm = (i != 0)                                   # FT tweak
        x = FTTransformerBlock(
            embed_dim=embedding_dim,                                    # D
            num_heads=num_heads,                                        # heads
            ff_dim=ff_dim,                                              # FF dim
            dropout_rate=dropout_rate,                                  # dropout
            attention_dropout=attention_dropout,                        # attn dropout
            use_attention_norm=use_attention_norm,                      # tweak
            name=f"ft_block_{i+1}"                                      # name
        )(x)
        print(f"  ✓ Transformer block {i+1}/{num_transformer_blocks} ready")  # log

    cls_rep = x[:, 0, :]                                                # CLS vector (B,D)

    print("  ✓ Building prediction head...")                             # log
    h = cls_rep                                                         # head input
    for j, units in enumerate(mlp_hidden_units):                        # MLP layers
        h = layers.Dense(units, activation="relu", name=f"mlp_dense_{j+1}")(h)  # dense
        h = layers.BatchNormalization(name=f"mlp_bn_{j+1}")(h)           # BN
        h = layers.Dropout(dropout_rate, name=f"mlp_drop_{j+1}")(h)      # dropout
        print(f"    - MLP layer {j+1}: {units} units")                   # log

    outputs = layers.Dense(num_outputs, activation="linear", name="output")(h)  # output

    model = Model(                                                      # build model
        inputs=[categorical_input, continuous_input],
        outputs=outputs,
        name="FTTransformer"
    )

    print("✓ FT-Transformer built successfully!")                        # log
    return model                                                        # return

In [18]:
# =========================
# CELL 19: Prepare Data for FT-Transformer (with diagnostics)
# =========================

print("Preparing data for FT-Transformer...")                            # log

categorical_cols = ['gender']                                            # categorical columns
continuous_cols = [c for c in X_train.columns if c not in categorical_cols]  # continuous columns

print(f"✓ Categorical features: {len(categorical_cols)} -> {categorical_cols}")  # log
print(f"✓ Continuous features: {len(continuous_cols)}")                   # log

X_train_cat = X_train[categorical_cols].values.astype('int32')            # cat train
X_train_cont = X_train[continuous_cols].values.astype('float32')          # cont train
X_val_cat = X_val[categorical_cols].values.astype('int32')                # cat val
X_val_cont = X_val[continuous_cols].values.astype('float32')              # cont val
Y_train_array = Y_train.values.astype('float32')                          # y train
Y_val_array = Y_val.values.astype('float32')                              # y val

print("\n✓ Data shapes:")                                                 # log
print(f"  Train: cat={X_train_cat.shape}, cont={X_train_cont.shape}, y={Y_train_array.shape}")  # log
print(f"  Val:   cat={X_val_cat.shape}, cont={X_val_cont.shape}, y={Y_val_array.shape}")        # log
print(f"  Weights: train={w_train.shape}, val={w_val.shape}")              # log

num_gender_categories = int(X_train['gender'].max()) + 1                   # gender vocab
categorical_cardinalities = [num_gender_categories]                        # vocab list
NUM_OUTPUTS = Y_train_array.shape[1]                                        # outputs

print(f"\n✓ categorical_cardinalities: {categorical_cardinalities}")       # log
print(f"✓ NUM_OUTPUTS: {NUM_OUTPUTS}")                                      # log

# ---- Attention memory estimator (why you saw ~26GB) ----
def estimate_attention_bytes(batch, heads, seq_len, bytes_per=4):
    return batch * heads * (seq_len * seq_len) * bytes_per                # approximate bytes

raw_seq_len = 1 + len(continuous_cols) + len(categorical_cols)            # original FT length
print(f"\n⚠️ Raw FT token length (no compression) L = {raw_seq_len}")      # log
print("   Attention is O(L^2). If L is huge, memory explodes.")            # log

Preparing data for FT-Transformer...
✓ Categorical features: 1 -> ['gender']
✓ Continuous features: 5121

✓ Data shapes:
  Train: cat=(4907, 1), cont=(4907, 5121), y=(4907, 14)
  Val:   cat=(1227, 1), cont=(1227, 5121), y=(1227, 14)
  Weights: train=(4907,), val=(1227,)

✓ categorical_cardinalities: [2]
✓ NUM_OUTPUTS: 14

⚠️ Raw FT token length (no compression) L = 5123
   Attention is O(L^2). If L is huge, memory explodes.


In [ ]:
# =========================
# CELL 20: Manual Grid Search (FT-Transformer) - memory safe
# =========================

from itertools import product                                             # cartesian product
from sklearn.model_selection import train_test_split                      # split
import gc                                                                 # garbage collect
import time                                                               # timing

print("\n==============================")
print("🚀 Starting Manual Grid Search (80/20 Split) - FT-Transformer")
print("==============================\n")

X_cat_tr, X_cat_val, X_cont_tr, X_cont_val, Y_tr, Y_val, W_tr, W_val = train_test_split(
    X_train_cat,                                                         # cat input
    X_train_cont,                                                        # cont input
    Y_train_array,                                                       # targets
    w_train,                                                             # weights
    test_size=0.2,                                                       # 80/20
    random_state=42                                                      # seed
)

print("✓ Data split complete:")
print(f"  Training samples: {len(X_cat_tr)}")
print(f"  Validation samples: {len(X_cat_val)}")

# ---- Smaller grid + new key params ----
param_grid = {
    "numeric_token_count": [16],                                  # KEY: numeric compression tokens
    "embedding_dim": [32,64,128],                                            # token dim
    "num_transformer_blocks": [2,4,8],                                     # depth
    "num_heads": [4,8,16],                                                  # heads
    "ff_dim": [128,256,512],                                                 # FF dim
    "mlp_hidden_units": [[256, 128]],                                     # head
    "dropout_rate": [0.1],                                                # dropout
    "attention_dropout": [0.1],                                           # attn dropout
    "learning_rate": [1e-5],                                        # lr
    "batch_size": [8]                                                 # smaller batch for CPU
}

keys, values = zip(*param_grid.items())                                   # unpack
combinations = [dict(zip(keys, v)) for v in product(*values)]             # combos

# Filter invalid head configs
combinations = [p for p in combinations if p["embedding_dim"] % p["num_heads"] == 0]

print(f"\n✅ Total parameter combinations: {len(combinations)}")

best_score = float("inf")                                                 # best loss
best_params = None                                                        # best params
start_time = time.time()                                                  # timer

for idx, params in enumerate(combinations):
    print("\n--------------------------------------------------")
    print(f"🔎 Combination {idx+1}/{len(combinations)}")
    print(params)
    print("--------------------------------------------------")

    # Compute effective sequence length with compression
    L = 1 + params["numeric_token_count"] + len(categorical_cols)          # CLS + K + n_cat
    approx_gb = estimate_attention_bytes(params["batch_size"], params["num_heads"], L) / (1024**3)
    print(f"🧠 Estimated attention matrix size ~ {approx_gb:.4f} GB (approx) for L={L}")  # log

    print("🏗 Building model...")
    model = build_fttransformer(
        num_categorical_features=len(categorical_cols),
        num_continuous_features=len(continuous_cols),
        categorical_cardinalities=categorical_cardinalities,
        embedding_dim=params["embedding_dim"],
        num_transformer_blocks=params["num_transformer_blocks"],
        num_heads=params["num_heads"],
        ff_dim=params["ff_dim"],
        mlp_hidden_units=params["mlp_hidden_units"],
        dropout_rate=params["dropout_rate"],
        attention_dropout=params["attention_dropout"],
        numeric_token_count=params["numeric_token_count"],
        num_outputs=NUM_OUTPUTS
    )

    model.compile(
        optimizer=Adam(learning_rate=params["learning_rate"]),
        loss="mse",
        metrics=["mae"]
    )

    early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=20,                                                      # faster stop for search
        restore_best_weights=True,
        verbose=1
    )

    reduce_lr = ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    )

    print("🏋 Training...")
    history = model.fit(
        [X_cat_tr, X_cont_tr],
        Y_tr,
        validation_data=([X_cat_val, X_cont_val], Y_val),
        sample_weight=W_tr,
        epochs=200,                                                       # fewer epochs for search
        batch_size=params["batch_size"],
        callbacks=[early_stopping, reduce_lr],
        verbose=1
    )

    val_loss = float(np.min(history.history["val_loss"]))
    print(f"📊 Validation Loss: {val_loss:.6f}")

    if val_loss < best_score:
        best_score = val_loss
        best_params = params
        print("⭐ NEW BEST MODEL FOUND!")

    tf.keras.backend.clear_session()
    gc.collect()

    elapsed = (time.time() - start_time) / 60
    print(f"⏱ Elapsed Time: {elapsed:.2f} minutes")

print("\n=================================")
print("🏆 GRID SEARCH COMPLETE")
print("=================================")
print(f"Best Parameters: {best_params}")
print(f"Best Validation Loss: {best_score:.6f}")


🚀 Starting Manual Grid Search (80/20 Split) - FT-Transformer

✓ Data split complete:
  Training samples: 3925
  Validation samples: 982

✅ Total parameter combinations: 81

--------------------------------------------------
🔎 Combination 1/81
{'numeric_token_count': 16, 'embedding_dim': 32, 'num_transformer_blocks': 2, 'num_heads': 4, 'ff_dim': 128, 'mlp_hidden_units': [256, 128], 'dropout_rate': 0.1, 'attention_dropout': 0.1, 'learning_rate': 1e-05, 'batch_size': 8}
--------------------------------------------------
🧠 Estimated attention matrix size ~ 0.0000 GB (approx) for L=18
🏗 Building model...

🏗 Building FT-Transformer...
  ✓ Tokenizing features...


2026-02-16 08:52:32.564465: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2026-02-16 08:52:32.564508: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:160] env: CUDA_VISIBLE_DEVICES="-1"
2026-02-16 08:52:32.564514: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:163] CUDA_VISIBLE_DEVICES is set to -1 - this hides all GPUs from CUDA
2026-02-16 08:52:32.564520: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:171] verbose logging is disabled. Rerun with verbose logging (usually --v=1 or --vmodule=cuda_diagnostics=1) to get more diagnostic output from this module
2026-02-16 08:52:32.564525: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:176] retrieving CUDA diagnostic information for host: c3316d4d8808
2026-02-16 08:52:32.564529: I external/local_xla/xla/stream_executor/cuda/

  ✓ Transformer block 1/2 ready
  ✓ Transformer block 2/2 ready
  ✓ Building prediction head...
    - MLP layer 1: 256 units
    - MLP layer 2: 128 units
✓ FT-Transformer built successfully!
🏋 Training...
Epoch 1/200


/usr/local/lib/python3.11/dist-packages/keras/src/optimizers/base_optimizer.py:870: UserWarning: Gradients do not exist for variables ['ft_block_1/ln1/gamma', 'ft_block_1/ln1/beta'] when minimizing the loss. If using `model.compile()`, did you forget to provide a `loss` argument?
  warnings.warn(


491/491 ━━━━━━━━━━━━━━━━━━━━ 13s 14ms/step - loss: 2.5400 - mae: 1.3752 - val_loss: 1.5386 - val_mae: 0.9895 - learning_rate: 1.0000e-05
Epoch 2/200
491/491 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - loss: 2.3410 - mae: 1.3154 - val_loss: 1.5020 - val_mae: 0.9795 - learning_rate: 1.0000e-05
Epoch 3/200
491/491 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - loss: 2.2063 - mae: 1.2722 - val_loss: 1.3760 - val_mae: 0.9406 - learning_rate: 1.0000e-05
Epoch 4/200
491/491 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - loss: 2.0089 - mae: 1.2254 - val_loss: 1.2575 - val_mae: 0.8977 - learning_rate: 1.0000e-05
Epoch 5/200
491/491 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - loss: 1.7896 - mae: 1.1803 - val_loss: 1.1250 - val_mae: 0.8460 - learning_rate: 1.0000e-05
Epoch 6/200
491/491 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - loss: 1.5692 - mae: 1.1281 - val_loss: 1.1005 - val_mae: 0.8328 - learning_rate: 1.0000e-05
Epoch 7/200
491/491 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - loss: 1.4176 - mae: 1.0981 - val_loss: 1.1265 - val_mae: 0.8360 - l

...

491/491 ━━━━━━━━━━━━━━━━━━━━ 48s 97ms/step - loss: 0.9518 - mae: 0.9683 - val_loss: 0.9964 - val_mae: 0.7891 - learning_rate: 2.5000e-06
Epoch 28/200
491/491 ━━━━━━━━━━━━━━━━━━━━ 47s 96ms/step - loss: 0.9587 - mae: 0.9656 - val_loss: 0.9719 - val_mae: 0.7776 - learning_rate: 2.5000e-06
Epoch 29/200
491/491 ━━━━━━━━━━━━━━━━━━━━ 48s 97ms/step - loss: 0.9447 - mae: 0.9615 - val_loss: 0.9878 - val_mae: 0.7865 - learning_rate: 2.5000e-06
Epoch 30/200
491/491 ━━━━━━━━━━━━━━━━━━━━ 48s 97ms/step - loss: 0.9418 - mae: 0.9578 - val_loss: 0.9577 - val_mae: 0.7715 - learning_rate: 2.5000e-06
Epoch 31/200
491/491 ━━━━━━━━━━━━━━━━━━━━ 47s 96ms/step - loss: 0.9380 - mae: 0.9543 - val_loss: 0.9494 - val_mae: 0.7694 - learning_rate: 2.5000e-06
Epoch 32/200
491/491 ━━━━━━━━━━━━━━━━━━━━ 47s 97ms/step - loss: 0.9211 - mae: 0.9486 - val_loss: 0.9434 - val_mae: 0.7658 - learning_rate: 2.5000e-06
Epoch 33/200
 45/491 ━━━━━━━━━━━━━━━━━━━━ 39s 89ms/step - loss: 0.9163 - mae: 0.9348

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



491/491 ━━━━━━━━━━━━━━━━━━━━ 48s 97ms/step - loss: 0.9111 - mae: 0.9481 - val_loss: 0.9114 - val_mae: 0.7499 - learning_rate: 2.5000e-06
Epoch 35/200
491/491 ━━━━━━━━━━━━━━━━━━━━ 47s 96ms/step - loss: 0.8891 - mae: 0.9325 - val_loss: 0.9241 - val_mae: 0.7539 - learning_rate: 2.5000e-06
Epoch 37/200
491/491 ━━━━━━━━━━━━━━━━━━━━ 48s 97ms/step - loss: 0.8738 - mae: 0.9230 - val_loss: 0.8710 - val_mae: 0.7310 - learning_rate: 2.5000e-06
Epoch 38/200
491/491 ━━━━━━━━━━━━━━━━━━━━ 47s 97ms/step - loss: 0.8650 - mae: 0.9243 - val_loss: 0.8698 - val_mae: 0.7309 - learning_rate: 2.5000e-06
Epoch 39/200
491/491 ━━━━━━━━━━━━━━━━━━━━ 48s 97ms/step - loss: 0.8592 - mae: 0.9236 - val_loss: 0.8569 - val_mae: 0.7256 - learning_rate: 2.5000e-06
Epoch 40/200
491/491 ━━━━━━━━━━━━━━━━━━━━ 47s 96ms/step - loss: 0.8493 - mae: 0.9175 - val_loss: 0.8524 - val_mae: 0.7220 - learning_rate: 2.5000e-06
Epoch 41/200
491/491 ━━━━━━━━━━━━━━━━━━━━ 47s 96ms/step - loss: 0.8387 - mae: 0.9066 - val_loss: 0.8388 - val_mae

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



491/491 ━━━━━━━━━━━━━━━━━━━━ 48s 97ms/step - loss: 0.8133 - mae: 0.8953 - val_loss: 0.8217 - val_mae: 0.7092 - learning_rate: 2.5000e-06
Epoch 45/200
491/491 ━━━━━━━━━━━━━━━━━━━━ 47s 96ms/step - loss: 0.8244 - mae: 0.8950 - val_loss: 0.8349 - val_mae: 0.7145 - learning_rate: 2.5000e-06
Epoch 47/200
491/491 ━━━━━━━━━━━━━━━━━━━━ 47s 96ms/step - loss: 0.8051 - mae: 0.8914 - val_loss: 0.8243 - val_mae: 0.7119 - learning_rate: 2.5000e-06
Epoch 48/200
491/491 ━━━━━━━━━━━━━━━━━━━━ 47s 96ms/step - loss: 0.7678 - mae: 0.8749 - val_loss: 0.8105 - val_mae: 0.7053 - learning_rate: 2.5000e-06
Epoch 49/200
491/491 ━━━━━━━━━━━━━━━━━━━━ 48s 97ms/step - loss: 0.7054 - mae: 0.8405 - val_loss: 0.7514 - val_mae: 0.6769 - learning_rate: 2.5000e-06
Epoch 59/200
491/491 ━━━━━━━━━━━━━━━━━━━━ 48s 99ms/step - loss: 0.7159 - mae: 0.8388 - val_loss: 0.7513 - val_mae: 0.6785 - learning_rate: 2.5000e-06
Epoch 60/200
491/491 ━━━━━━━━━━━━━━━━━━━━ 49s 100ms/step - loss: 0.7046 - mae: 0.8370 - val_loss: 0.7828 - val_ma

In [ ]:
# =========================
# CELL 21: Train Final Best FT-Transformer
# =========================

print("\n==============================")                                  # log
print("🚀 Training Final Best FT-Transformer")                           # log
print("==============================\n")                                  # log

if best_params is None:
    raise ValueError("❌ Run Grid Search first!")                        # safety check

print("🏆 Best Parameters:")                                             # log
print(best_params)                                                      # log

final_model = build_fttransformer(
    num_categorical_features=len(categorical_cols),                     # cat count
    num_continuous_features=len(continuous_cols),                       # cont count
    categorical_cardinalities=categorical_cardinalities,                # vocab sizes
    embedding_dim=best_params["embedding_dim"],                         # best embed dim
    num_transformer_blocks=best_params["num_transformer_blocks"],       # best blocks
    num_heads=best_params["num_heads"],                                 # best heads
    ff_dim=best_params["ff_dim"],                                       # best FFN dim
    mlp_hidden_units=best_params["mlp_hidden_units"],                   # best MLP
    dropout_rate=best_params["dropout_rate"],                           # best dropout
    attention_dropout=best_params["attention_dropout"],                 # best attn dropout
    num_outputs=NUM_OUTPUTS                                             # outputs
)

final_model.compile(
    optimizer=Adam(learning_rate=best_params["learning_rate"]),         # best LR
    loss="mse",                                                         # loss
    metrics=["mae"]                                                     # metric
)

history = final_model.fit(
    [X_train_cat, X_train_cont],                                        # train inputs
    Y_train_array,                                                      # train targets
    validation_data=([X_val_cat, X_val_cont], Y_val_array),             # validation inputs
    sample_weight=w_train,                                              # weights
    epochs=200,                                                         # epochs
    batch_size=64,                                                      # batch
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True),  # early stop
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)   # LR reduction
    ],
    verbose=1                                                           # verbose
)

print("✅ Final model training complete.")                               # log

In [ ]:
# =========================
# CELL 22: Save Final Model (.h5)
# =========================

print("💾 Saving model as tabtransformerBest.h5 ...")                    # keep same name as your original pipeline
final_model.save("ft-transformerBest-v20.h5")                               # save model
print("✅ Model saved successfully!")                                    # log

In [ ]:
# =========================
# CELL 22B: Save Final Model (.keras)
# =========================

print("💾 Saving model as tabtransformerBest.keras ...")                 # keep same name as your original pipeline
final_model.save("ft-transformerBest-v20.keras")                            # save model
print("✅ Model saved successfully!")                                    # log
print("📁 File created: tabtransformerBest.keras")                       # log

In [ ]:
# =========================
# CELL 23: Plot Training History
# =========================

print("📊 Plotting training history...")                                 # log

fig, axes = plt.subplots(1, 2, figsize=(15, 5))                          # create plots

axes[0].plot(history.history['loss'], label='Train Loss', linewidth=2)   # train loss
axes[0].plot(history.history['val_loss'], label='Val Loss', linewidth=2) # val loss

best_epoch = np.argmin(history.history['val_loss'])                      # best epoch index

axes[0].scatter(
    best_epoch,
    history.history['val_loss'][best_epoch],
    color='red',
    s=100,
    label='Best Epoch'
)

axes[0].set_title('Model Loss', fontsize=14, fontweight='bold')          # title
axes[0].set_xlabel('Epoch')                                              # x label
axes[0].set_ylabel('MSE')                                                # y label
axes[0].legend()                                                         # legend
axes[0].grid(True, alpha=0.3)                                            # grid

axes[1].plot(history.history['mae'], label='Train MAE', linewidth=2)      # train mae
axes[1].plot(history.history['val_mae'], label='Val MAE', linewidth=2)    # val mae

axes[1].scatter(
    best_epoch,
    history.history['val_mae'][best_epoch],
    color='red',
    s=100,
    label='Best Epoch'
)

axes[1].set_title('Model MAE', fontsize=14, fontweight='bold')            # title
axes[1].set_xlabel('Epoch')                                              # x label
axes[1].set_ylabel('MAE')                                                # y label
axes[1].legend()                                                         # legend
axes[1].grid(True, alpha=0.3)                                            # grid

plt.tight_layout()                                                       # layout
plt.savefig('training_history.png', dpi=300, bbox_inches='tight')         # save plot
plt.show()                                                               # show plot

print("\n✓ Training Summary:")                                           # summary
print(f"  Total epochs trained: {len(history.history['loss'])}")         # epochs count
print(f"  Best epoch: {best_epoch + 1}")                                 # best epoch
print(f"  Best train loss: {history.history['loss'][best_epoch]:.4f}")   # best train loss
print(f"  Best val loss: {history.history['val_loss'][best_epoch]:.4f}") # best val loss
print(f"  Best train MAE: {history.history['mae'][best_epoch]:.4f}")     # best train MAE
print(f"  Best val MAE: {history.history['val_mae'][best_epoch]:.4f}")   # best val MAE

In [ ]:
# =========================
# CELL 24: Evaluate Final Model
# =========================

print("🔍 Evaluating final best model...")                               # log

if 'final_model' not in globals():                                      # safety check
    raise ValueError("❌ final_model not found. Run Cell 21 first.")     # error

val_loss, val_mae = final_model.evaluate(
    [X_val_cat, X_val_cont],                                            # inputs
    Y_val_array,                                                        # targets
    batch_size=64,                                                      # batch size
    verbose=1                                                           # verbose
)

print(f"\n✓ Validation metrics:")                                       # log
print(f"  Loss (MSE): {val_loss:.4f}")                                   # loss
print(f"  MAE: {val_mae:.4f}")                                           # mae

print("\n📈 Generating predictions...")                                  # log

predictions = final_model.predict(
    [X_val_cat, X_val_cont],                                            # inputs
    batch_size=64                                                       # batch size
)

print("\n✓ Per-target MAE:")                                             # log

target_names = Y_train.columns.tolist()                                 # target names

for i, name in enumerate(target_names):                                 # loop outputs
    mae = np.mean(np.abs(predictions[:, i] - Y_val_array[:, i]))         # compute MAE
    print(f"  {name:25s}: {mae:.4f}")                                    # print MAE

print("\n✅ Evaluation complete!")                                       # done